# Neo4j Query Graph Pipeline

In [455]:
%pip -q install python-dotenv

from dotenv import load_dotenv
import os

load_dotenv(".env")   # подхватит /.../neo4j_граф/.env

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_KEY = os.getenv("GPT_API_KEY")

print("DEEPSEEK_API_KEY loaded:", bool(DEEPSEEK_API_KEY))

Note: you may need to restart the kernel to use updated packages.
DEEPSEEK_API_KEY loaded: True


In [456]:
# 1) Config and imports
import os
import re
import json
from pathlib import Path
import requests
from neo4j import GraphDatabase

NEO4J_URI = os.getenv('NEO4J_URI', 'neo4j://localhost:7689')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '12345678')
NEO4J_DB = os.getenv('NEO4J_DB', 'neo4j')

GPT_API_KEY = os.getenv('GPT_API_KEY')
GPT_MODEL = os.getenv('GPT_MODEL', 'gpt-5.4-mini')

BASE_DIR = Path.cwd()
ANNOT_DIR = BASE_DIR / 'annotation'
ANNOT_QG_DIR = BASE_DIR / 'annotaion_q_graph'


In [457]:
# 2) Neo4j helpers and candidate lists
def run_cypher(query, params=None):
    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as _driver:
        with _driver.session(database=NEO4J_DB) as _session:
            return [r.data() for r in _session.run(query, params or {})]

def has_label(label):
    labels = [r['label'] for r in run_cypher('CALL db.labels() YIELD label RETURN label') if r.get('label')]
    return label in set(labels)

def has_property(prop):
    props = [r['propertyKey'] for r in run_cypher('CALL db.propertyKeys() YIELD propertyKey RETURN propertyKey') if r.get('propertyKey')]
    return prop in set(props)

def fetch_candidates():
    cities = [r['key'] for r in run_cypher('MATCH (c:City) RETURN c.key AS key') if r.get('key')]
    airports = [r['key'] for r in run_cypher('MATCH (a:Airport) RETURN a.key AS key') if r.get('key')]
    airport_city_rows = run_cypher("MATCH (a:Airport)-[:locatedInCity]->(c:City) RETURN a.key AS airport, c.key AS city")
    city_airports = {}
    for r in airport_city_rows:
        a = r.get('airport'); c = r.get('city')
        if not a or not c: continue
        city_airports.setdefault(c, set()).add(a)
    countries = [r['key'] for r in run_cypher('MATCH (c:Country) RETURN c.key AS key') if r.get('key')]
    doc_types = [r['v'] for r in run_cypher('MATCH (di:DocumentInstance) WHERE di.documentType IS NOT NULL RETURN DISTINCT di.documentType AS v') if r.get('v')]
    topic_keys = [r['v'] for r in run_cypher('MATCH (q:Questioning) WHERE q.topicKey IS NOT NULL RETURN DISTINCT q.topicKey AS v') if r.get('v')]
    biometric = []
    if has_label('BiometricCheck') and has_property('biometricModality'):
        biometric = [r['v'] for r in run_cypher('MATCH (b:BiometricCheck) WHERE b.biometricModality IS NOT NULL RETURN DISTINCT b.biometricModality AS v') if r.get('v')]
    return {
        'cities': sorted(cities),
        'airports': sorted(airports),
        'countries': sorted(countries),
        'doc_types': sorted(doc_types),
        'topic_keys': sorted(topic_keys),
        'biometric_modalities': sorted(biometric),
        'city_airports': {k: sorted(list(v)) for k, v in city_airports.items()},
    }

candidates = fetch_candidates()



In [458]:
# 3) Load question text
def get_question_text(qnum: int) -> str:
    base = ANNOT_QG_DIR / f'q_{qnum}'
    for folder in ['fr_1', 'fr_2', 'germany', 'italy', 'spain']:
        p = base / folder
        if not p.exists():
            continue
        for f in sorted(p.glob('annotation_*.json')):
            data = json.loads(f.read_text(encoding='utf-8'))
            q = (data.get('question') or '').strip()
            if q:
                return q
    raise FileNotFoundError(f'Question text not found for q_{qnum}')

# Provide a list of question numbers here
QNUMS = [i for i in range(1, 64)]
QUESTIONS = {q: get_question_text(q) for q in QNUMS}

# Default single-run variables (first in list)
QNUM = QNUMS[0]
QUESTION = QUESTIONS[QNUM]
QUESTION


'Which documents do I need to bring to passport control in Paris?'

In [459]:
# 4) LLM router: intents, mode, anchors
def llm_route(question: str, candidates: dict) -> dict:
    if not GPT_API_KEY:
        return {
            'intents': ['documentCheck'],
            'mode': 'single',
            'conditional': None,
            'scope': {'cities': [], 'airports': [], 'countries': []},
            'exclude_scope': {'cities': [], 'airports': [], 'countries': []},
            'doc_types': [],
            'topic_keys': [],
            'biometric_modalities': [],
            'aggregate': None,
        }

    system_parts = [
        'You route a question to intents and retrieval mode.',
        'Return ONLY JSON with keys: intents, mode, conditional, scope, exclude_scope, doc_types, topic_keys, biometric_modalities, aggregate.',
        'Select anchors ONLY from provided candidates.',
        'Only include city/airport if explicitly named in the question (not just the word city).',
        'If Paris is explicitly mentioned, ALWAYS include Beauvais city and airport BVA.',
        'Only include topic_keys if a specific topic is explicitly mentioned in the question (not just the phrase question topics).',
        'Only include biometric_modalities if a specific modality is explicitly mentioned (e.g., fingerprint, face photo).',
        'If no specific topic is explicitly mentioned, topic_keys MUST be [].',
        'If no specific biometric modality is explicitly mentioned, biometric_modalities MUST be [].',
        'Do NOT auto-fill topic_keys or biometric_modalities from candidates.',
        'Choose intents from: documentCheck, questioning, biometric.',
        'mode must be one of: single, union, intersect, conditional, aggregate, difference.',
        'If mode=conditional, set conditional to doc_given_topic or topic_given_doc.',
        'If the question asks which documents are checked for travelers who were asked about topic X, use mode=conditional and conditional=doc_given_topic and put X in topic_keys.',
        'If the question asks which question topics are asked of travelers whose documents include type X (e.g., visa), use mode=conditional and conditional=topic_given_doc and put X in doc_types.',
        'If the question asks for A but not B (e.g., in Paris but not in Nice), set mode=difference and put A in scope and B in exclude_scope.',
        'If the question is about most/least/how many/how often/top/fewest, you MUST set mode=aggregate and fill aggregate.',
        'If a biometric modality is explicitly mentioned and the question is aggregate (e.g., how often), you MUST include intent biometric and set aggregate.metric=biometric.',
        'If the scope contains only countries (no cities/airports) for an aggregate biometric question, set aggregate.group_by=country.',
        "If the question compares which question topic is more common (e.g., cash vs card), use intents=['questioning'], mode=aggregate, aggregate.group_by=topic, metric=questions, top_k=1, and put those topics in topic_keys. If the question uses general words like cash/card and no topic keys are explicitly mentioned, you MUST choose the closest available topic keys from candidates (e.g., cash -> cash_amount, card -> payment_card).",
        'If the question is about arrivals/entry points/most often fly into, use aggregate.metric=encounters.',
        'aggregate.group_by must be country/city/airport/topic, metric one of documents/questions/biometric/encounters/no_questions/passport_only, order asc/desc, top_k integer.',
        'If not aggregate, set aggregate to null. If not difference, set exclude_scope to empty arrays.'
    ]
    system = ' '.join(system_parts)



    user = {
        'question': question,
        'candidates': candidates
    }

    schema = {
        'type': 'object',
        'additionalProperties': False,
        'required': ['intents','mode','conditional','scope','exclude_scope','doc_types','topic_keys','biometric_modalities','aggregate'],
        'properties': {
            'intents': {'type': 'array', 'items': {'type': 'string'}},
            'mode': {'type': 'string'},
            'conditional': {'type': 'string'},
            'scope': {
                'type': 'object',
                'additionalProperties': False,
                'required': ['cities','airports','countries'],
                'properties': {
                    'cities': {'type': 'array', 'items': {'type': 'string'}},
                    'airports': {'type': 'array', 'items': {'type': 'string'}},
                    'countries': {'type': 'array', 'items': {'type': 'string'}}
                }
            },
            'exclude_scope': {
                'type': 'object',
                'additionalProperties': False,
                'required': ['cities','airports','countries'],
                'properties': {
                    'cities': {'type': 'array', 'items': {'type': 'string'}},
                    'airports': {'type': 'array', 'items': {'type': 'string'}},
                    'countries': {'type': 'array', 'items': {'type': 'string'}}
                }
            },
            'doc_types': {'type': 'array', 'items': {'type': 'string'}},
            'topic_keys': {'type': 'array', 'items': {'type': 'string'}},
            'biometric_modalities': {'type': 'array', 'items': {'type': 'string'}},
            'aggregate': {
                'anyOf': [
                    {'type': 'null'},
                    {
                        'type': 'object',
                        'additionalProperties': False,
                        'required': ['group_by','metric','order','top_k'],
                        'properties': {
                            'group_by': {'type': 'string'},
                            'metric': {'type': 'string'},
                            'order': {'type': 'string'},
                            'top_k': {'type': 'integer'}
                        }
                    }
                ]
            }
        }
    }

    payload = {
        'model': GPT_MODEL,
        'input': [
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': json.dumps(user, ensure_ascii=False)}
        ],
        'text': {
            'format': {
                'type': 'json_schema',
                'name': 'route',
                'schema': schema,
                'strict': True
            }
        }
    }

    resp = requests.post(
        'https://api.openai.com/v1/responses',
        headers={'Authorization': f'Bearer {GPT_API_KEY}'},
        json=payload,
        timeout=60
    )
    if not resp.ok:
        raise RuntimeError(f'OpenAI API error {resp.status_code}: {resp.text[:500]}')
    data = resp.json()
    text = data.get('output_text') or ''
    if not text:
        out = []
        for item in data.get('output', []):
            for content in item.get('content', []):
                if content.get('type') == 'output_text':
                    out.append(content.get('text', ''))
        text = ''.join(out).strip()
    if not text:
        raise ValueError('Empty LLM response')
    return json.loads(text)

route = llm_route(QUESTION, candidates)
route






{'intents': ['documentCheck'],
 'mode': 'single',
 'conditional': '',
 'scope': {'cities': ['city_paris', 'city_beauvais'],
  'airports': ['airport_bva'],
  'countries': []},
 'exclude_scope': {'cities': [], 'airports': [], 'countries': []},
 'doc_types': [],
 'topic_keys': [],
 'biometric_modalities': [],
 'aggregate': None}

In [460]:
# 5) Sanitize route output
def sanitize_route(route, candidates):
    allowed_intents = {'documentCheck','questioning','biometric'}
    allowed_modes = {'single','union','intersect','conditional','aggregate','difference'}
    allowed_conditional = {'doc_given_topic','topic_given_doc'}
    allowed_metric = {'documents','questions','biometric','encounters','no_questions','passport_only'}
    allowed_group = {'country','city','airport','topic'}
    allowed_order = {'asc','desc'}

    out = {
        'intents': [],
        'mode': route.get('mode','single'),
        'conditional': route.get('conditional'),
        'scope': {'cities': [], 'airports': [], 'countries': []},
        'exclude_scope': {'cities': [], 'airports': [], 'countries': []},
        'doc_types': [],
        'topic_keys': [],
        'biometric_modalities': [],
        'aggregate': route.get('aggregate')
    }

    intents = route.get('intents') if isinstance(route, dict) else []
    out['intents'] = [i for i in intents if i in allowed_intents] or ['documentCheck']

    scope = route.get('scope', {}) if isinstance(route, dict) else {}
    for k in ['cities','airports','countries']:
        vals = scope.get(k, []) if isinstance(scope, dict) else []
        allowed = set(candidates.get(k, []))
        out['scope'][k] = sorted({v for v in vals if v in allowed})

    # Paris implies Beauvais + BVA
    if 'city_paris' in out['scope']['cities']:
        if 'city_beauvais' in set(candidates.get('cities', [])):
            out['scope']['cities'] = sorted(set(out['scope']['cities']) | {'city_beauvais'})
        if 'airport_bva' in set(candidates.get('airports', [])):
            out['scope']['airports'] = sorted(set(out['scope']['airports']) | {'airport_bva'})

    ex_scope = route.get('exclude_scope', {}) if isinstance(route, dict) else {}
    for k in ['cities','airports','countries']:
        vals = ex_scope.get(k, []) if isinstance(ex_scope, dict) else []
        allowed = set(candidates.get(k, []))
        out['exclude_scope'][k] = sorted({v for v in vals if v in allowed})

    # For difference: move excluded cities out of scope and add their airports to exclude_scope
    if out['mode'] == 'difference':
        ex_cities = set(out['exclude_scope']['cities'])
        if ex_cities:
            out['scope']['cities'] = sorted(set(out['scope']['cities']) - ex_cities)
            city_airports = candidates.get('city_airports', {})
            ex_airports = set(out['exclude_scope']['airports'])
            for c in ex_cities:
                for a in city_airports.get(c, []):
                    ex_airports.add(a)
            out['exclude_scope']['airports'] = sorted(ex_airports)
            out['scope']['airports'] = sorted(set(out['scope']['airports']) - ex_airports)
    for k in ['doc_types','topic_keys','biometric_modalities']:
        vals = route.get(k, []) if isinstance(route, dict) else []
        allowed = set(candidates.get(k, []))
        out[k] = sorted({v for v in vals if v in allowed})

    if out['mode'] not in allowed_modes:
        out['mode'] = 'single'

    if out['mode'] == 'union' and len(out['intents']) < 2:
        out['mode'] = 'single'

    if out['mode'] == 'difference':
        if 'documentCheck' not in set(out['intents']):
            out['mode'] = 'single'
        if not any(out['exclude_scope'].values()):
            out['mode'] = 'single'

    if out['mode'] in {'conditional','intersect'}:
        if not ({'documentCheck','questioning'} <= set(out['intents'])):
            out['mode'] = 'single'

    if out['mode'] == 'conditional':
        if out['conditional'] not in allowed_conditional:
            out['conditional'] = 'doc_given_topic'

    agg = out.get('aggregate')
    if out['mode'] != 'aggregate':
        out['aggregate'] = None
    else:
        if not isinstance(agg, dict):
            out['mode'] = 'single'
            out['aggregate'] = None
        else:
            group_by = agg.get('group_by')
            metric = agg.get('metric')
            order = agg.get('order','desc')
            top_k = int(agg.get('top_k', 1))
            intents_set = set(out.get('intents') or [])
            if metric != 'encounters':
                if 'biometric' in intents_set:
                    metric = 'biometric'
                elif 'questioning' in intents_set and 'documentCheck' not in intents_set:
                    metric = 'questions'
                elif 'documentCheck' in intents_set and 'questioning' not in intents_set:
                    metric = 'documents'
            if metric == 'encounters':
                out['intents'] = ['documentCheck']
            if metric == 'no_questions':
                out['intents'] = ['questioning']
            if metric == 'passport_only':
                out['intents'] = ['documentCheck','questioning']
            if group_by == 'topic':
                metric = 'questions'
                out['intents'] = ['questioning']
            if group_by not in allowed_group or metric not in allowed_metric:
                out['mode'] = 'single'
                out['aggregate'] = None
            else:
                out['aggregate'] = {
                    'group_by': group_by,
                    'metric': metric,
                    'order': order if order in allowed_order else 'desc',
                    'top_k': max(1, top_k)
                }

    return out

ROUTE = sanitize_route(route, candidates)

def apply_heuristics(question: str, route: dict, candidates: dict) -> dict:
    q = (question or '').lower()

    def mentions_any_city(qtext: str) -> bool:
        for key in candidates.get('cities', []):
            name = key.replace('city_', '').replace('_', ' ')
            if name and name in qtext:
                return True
        return False

    def mentions_any_airport(qtext: str) -> bool:
        for key in candidates.get('airports', []):
            name = key.replace('airport_', '').replace('_', ' ')
            if name and (name in qtext or (name + 's') in qtext):
                return True
        return False

    def mentions_any_topic(qtext: str) -> bool:
        for key in candidates.get('topic_keys', []):
            name = key.replace('_', ' ')
            if name and (name in qtext or (name + 's') in qtext):
                return True
        return False

    def mentions_any_biometric(qtext: str) -> bool:
        for key in candidates.get('biometric_modalities', []):
            name = key.replace('_', ' ')
            if name and (name in qtext or (name + 's') in qtext):
                return True
        return False

    # Force aggregate for 'most common question topics'

    # Force single for 'which question topics are asked' (non-aggregate)
    if 'question topic' in q and not any(x in q for x in ['most common', 'most often', 'most frequent', 'most', 'least', 'fewest', 'how many', 'how often', 'top']):
        route['intents'] = ['questioning']
        route['mode'] = 'single'
        route['aggregate'] = None
    if any(x in q for x in ['most common', 'most often', 'most frequent']) and 'question topic' in q:
        route['intents'] = ['questioning']
        route['mode'] = 'aggregate'
        route['aggregate'] = {
            'group_by': 'topic',
            'metric': 'questions',
            'order': 'desc',
            'top_k': 10
        }

    # Force encounters metric for arrivals/entry point questions
    if route.get('mode') == 'aggregate' and any(x in q for x in ['fly into', 'fly in', 'entry point', 'arrive in', 'arrival']):
        agg = route.get('aggregate') or {}
        agg['metric'] = 'encounters'
        route['aggregate'] = agg

    # No-questions aggregate
    if any(x in q for x in ['no questions', 'without questions', 'no questioning']):
        route['intents'] = ['questioning']
        route['mode'] = 'aggregate'
        route['aggregate'] = {
            'group_by': 'country',
            'metric': 'no_questions',
            'order': 'desc',
            'top_k': 1
        }

    # Passport check only (no questions + passport document only)
    if 'passport check' in q:
        route['intents'] = ['documentCheck','questioning']
        route['mode'] = 'aggregate'
        route['aggregate'] = {
            'group_by': 'country',
            'metric': 'passport_only',
            'order': 'desc',
            'top_k': 1
        }
        if not route.get('doc_types'):
            route['doc_types'] = ['foreign_passport']

    # If question asks 'which city in <country>' without naming a city, clear city/airport filters
    if 'which city' in q and not mentions_any_city(q):
        route['scope']['cities'] = []
        route['scope']['airports'] = []

    # If no airport explicitly mentioned, clear scope airports (except Paris->BVA)
    if not mentions_any_airport(q):
        route['scope']['airports'] = []
        if 'city_paris' in route['scope']['cities'] and 'airport_bva' in candidates.get('airports', []):
            route['scope']['airports'] = sorted(set(route['scope']['airports']) | {'airport_bva'})

    return route

ROUTE = apply_heuristics(QUESTION, ROUTE, candidates)
ROUTE










{'intents': ['documentCheck'],
 'mode': 'single',
 'conditional': '',
 'scope': {'cities': ['city_beauvais', 'city_paris'],
  'airports': ['airport_bva'],
  'countries': []},
 'exclude_scope': {'cities': [], 'airports': [], 'countries': []},
 'doc_types': [],
 'topic_keys': [],
 'biometric_modalities': [],
 'aggregate': None}

In [461]:
# 6) Build query graph object
QUERY_GRAPH = {
    'question': QUESTION,
    **ROUTE
}
QUERY_GRAPH


{'question': 'Which documents do I need to bring to passport control in Paris?',
 'intents': ['documentCheck'],
 'mode': 'single',
 'conditional': '',
 'scope': {'cities': ['city_beauvais', 'city_paris'],
  'airports': ['airport_bva'],
  'countries': []},
 'exclude_scope': {'cities': [], 'airports': [], 'countries': []},
 'doc_types': [],
 'topic_keys': [],
 'biometric_modalities': [],
 'aggregate': None}

In [462]:
# 7) Compile query graph to Cypher
SCOPE_MATCH = '''
MATCH (e:Encounter)
OPTIONAL MATCH (e)-[:atAirport]->(a:Airport)
OPTIONAL MATCH (e)-[:atCity]->(c:City)
OPTIONAL MATCH (e)-[:atCountry]->(co:Country)
OPTIONAL MATCH (a)-[:locatedInCity]->(c2:City)
OPTIONAL MATCH (a)-[:locatedInCountry]->(co2:Country)
OPTIONAL MATCH (c)-[:locatedInCountry]->(co3:Country)
WITH e,a,c,co,c2,co2,co3,
     (size($airports) > 0) AS has_airports,
     (size($cities) > 0) AS has_cities,
     (size($countries) > 0) AS has_countries,
     (a IS NOT NULL AND a.key IN $airports) AS match_airport,
     ((c IS NOT NULL AND c.key IN $cities) OR (c2 IS NOT NULL AND c2.key IN $cities)) AS match_city,
     ((co IS NOT NULL AND co.key IN $countries) OR (co2 IS NOT NULL AND co2.key IN $countries) OR (co3 IS NOT NULL AND co3.key IN $countries)) AS match_country
WHERE CASE
    WHEN (has_airports OR has_cities) THEN (match_airport OR match_city)
    WHEN has_countries THEN match_country
    ELSE true
END
'''

SCOPE_MATCH_EXCLUDE = '''
MATCH (e:Encounter)
OPTIONAL MATCH (e)-[:atAirport]->(a:Airport)
OPTIONAL MATCH (e)-[:atCity]->(c:City)
OPTIONAL MATCH (e)-[:atCountry]->(co:Country)
OPTIONAL MATCH (a)-[:locatedInCity]->(c2:City)
OPTIONAL MATCH (a)-[:locatedInCountry]->(co2:Country)
OPTIONAL MATCH (c)-[:locatedInCountry]->(co3:Country)
WITH e,a,c,co,c2,co2,co3,
     (size($airports) > 0) AS has_airports,
     (size($cities) > 0) AS has_cities,
     (size($countries) > 0) AS has_countries,
     (a IS NOT NULL AND a.key IN $airports) AS match_airport,
     ((c IS NOT NULL AND c.key IN $cities) OR (c2 IS NOT NULL AND c2.key IN $cities)) AS match_city,
     ((co IS NOT NULL AND co.key IN $countries) OR (co2 IS NOT NULL AND co2.key IN $countries) OR (co3 IS NOT NULL AND co3.key IN $countries)) AS match_country,
     (size($ex_airports) > 0) AS has_ex_airports,
     (size($ex_cities) > 0) AS has_ex_cities,
     (size($ex_countries) > 0) AS has_ex_countries,
     (a IS NOT NULL AND a.key IN $ex_airports) AS match_ex_airport,
     ((c IS NOT NULL AND c.key IN $ex_cities) OR (c2 IS NOT NULL AND c2.key IN $ex_cities)) AS match_ex_city,
     ((co IS NOT NULL AND co.key IN $ex_countries) OR (co2 IS NOT NULL AND co2.key IN $ex_countries) OR (co3 IS NOT NULL AND co3.key IN $ex_countries)) AS match_ex_country
WHERE CASE
    WHEN (has_airports OR has_cities) THEN (match_airport OR match_city)
    WHEN has_countries THEN match_country
    ELSE true
END
AND NOT (CASE
    WHEN (has_ex_airports OR has_ex_cities) THEN (match_ex_airport OR match_ex_city)
    WHEN has_ex_countries THEN match_ex_country
    ELSE false
END)
'''

def compile_aggregate_cypher(qg):
    agg = qg.get('aggregate') or {}
    group_by = agg.get('group_by')
    metric = agg.get('metric')
    order = agg.get('order','desc')
    top_k = int(agg.get('top_k', 1))
    limit_clause = f"LIMIT {top_k}" if top_k > 0 else ''

    if group_by == 'topic':
        if metric != 'questions':
            raise ValueError('Unsupported aggregate metric for topic')
        return f'''
{SCOPE_MATCH}
MATCH (e)-[:hasQuestioning]->(q:Questioning)
WHERE q.topicKey IS NOT NULL AND (size($topic_keys)=0 OR q.topicKey IN $topic_keys)
RETURN q.topicKey AS topic, count(*) AS metric
ORDER BY metric {order}, topic ASC
{limit_clause}
'''.strip()

    if group_by == 'country':
        group_expr = 'coalesce(co, co2, co3)'
        group_key = 'country.key'
        group_with = f"WITH e,a,c,co,c2,co2,co3, {group_expr} AS country"
        group_where = 'WHERE country IS NOT NULL'
        group_name = 'country'
    elif group_by == 'city':
        group_expr = 'coalesce(c, c2)'
        group_key = 'city.key'
        group_with = f"WITH e,a,c,co,c2,co2,co3, {group_expr} AS city"
        group_where = 'WHERE city IS NOT NULL'
        group_name = 'city'
    elif group_by == 'airport':
        group_expr = 'a'
        group_key = 'airport.key'
        group_with = f"WITH e,a,c,co,c2,co2,co3, {group_expr} AS airport"
        group_where = 'WHERE airport IS NOT NULL'
        group_name = 'airport'
    else:
        raise ValueError('Unsupported aggregate group_by')

    if metric == 'documents':
        metric_match = 'MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)'
        metric_where = 'WHERE (size($doc_types)=0 OR di.documentType IN $doc_types)'
        metric_expr = 'count(*) AS metric'
    elif metric == 'questions':
        metric_match = 'MATCH (e)-[:hasQuestioning]->(q:Questioning)'
        metric_where = 'WHERE (size($topic_keys)=0 OR q.topicKey IN $topic_keys)'
        metric_expr = 'count(*) AS metric'
    elif metric == 'biometric':
        metric_match = 'MATCH (e)-[:hasBiometricCheck]->(b:BiometricCheck)'
        metric_where = 'WHERE (size($biometric_modalities)=0 OR b.biometricModality IN $biometric_modalities)'
        metric_expr = 'count(*) AS metric'
    elif metric == 'encounters':
        metric_match = ''
        metric_where = ''
        metric_expr = 'count(DISTINCT e) AS metric'
    elif metric == 'no_questions':
        return f'''
{SCOPE_MATCH}
{group_with}
{group_where}
OPTIONAL MATCH (e)-[:hasQuestioning]->(q:Questioning)
WITH {group_name}, e, count(q) AS qcnt
WHERE qcnt = 0
RETURN {group_key} AS {group_name}, count(DISTINCT e) AS metric
ORDER BY metric {order}, {group_name} ASC
{limit_clause}
'''.strip()
    elif metric == 'passport_only':
        return f'''
{SCOPE_MATCH}
{group_with}
{group_where}
OPTIONAL MATCH (e)-[:hasQuestioning]->(q:Questioning)
WITH {group_name}, e, count(q) AS qcnt
WHERE qcnt = 0
MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)
WITH {group_name}, e, collect(DISTINCT di.documentType) AS dtypes
WHERE size(dtypes)=1 AND ((size($doc_types)=0 AND 'foreign_passport' IN dtypes) OR (size($doc_types)>0 AND dtypes[0] IN $doc_types))
RETURN {group_key} AS {group_name}, count(DISTINCT e) AS metric
ORDER BY metric {order}, {group_name} ASC
{limit_clause}
'''.strip()
    else:
        raise ValueError('Unsupported aggregate metric')

    return f'''
{SCOPE_MATCH}
{group_with}
{group_where}
{metric_match}
{metric_where}
RETURN {group_key} AS {group_name}, {metric_expr}
ORDER BY metric {order}, {group_name} ASC
{limit_clause}
'''.strip()


def compile_documentcheck_cypher(qg):
    return f'''
{SCOPE_MATCH}
OPTIONAL MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)
OPTIONAL MATCH (dc)-[:requestedDocument]->(di:DocumentInstance)
WITH e,a,c,co,c2,co2,co3,dc,di,di.documentType AS di_type
WHERE (size($doc_types)=0 OR di_type IN $doc_types)
CALL (e,dc,di,di_type,a,c,co,c2,co2,co3) {{
  WITH e,dc WHERE e IS NOT NULL AND dc IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasDocumentCheck' AS p, labels(dc)[0] AS o_label, dc.key AS o_key
  UNION
  WITH dc,di WHERE dc IS NOT NULL AND di IS NOT NULL
  RETURN DISTINCT labels(dc)[0] AS s_label, dc.key AS s_key, 'requestedDocument' AS p, labels(di)[0] AS o_label, di.key AS o_key
  UNION
  WITH di,di_type WHERE di IS NOT NULL AND di_type IS NOT NULL
  RETURN DISTINCT labels(di)[0] AS s_label, di.key AS s_key, 'documentType' AS p, 'Literal' AS o_label, di_type AS o_key
  UNION
  WITH e,c WHERE e IS NOT NULL AND c IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCity' AS p, labels(c)[0] AS o_label, c.key AS o_key
  UNION
  WITH e,a WHERE e IS NOT NULL AND a IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atAirport' AS p, labels(a)[0] AS o_label, a.key AS o_key
  UNION
  WITH e,co WHERE e IS NOT NULL AND co IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCountry' AS p, labels(co)[0] AS o_label, co.key AS o_key
  UNION
  WITH a,c2 WHERE a IS NOT NULL AND c2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCity' AS p, labels(c2)[0] AS o_label, c2.key AS o_key
  UNION
  WITH c,co3 WHERE c IS NOT NULL AND co3 IS NOT NULL
  RETURN DISTINCT labels(c)[0] AS s_label, c.key AS s_key, 'locatedInCountry' AS p, labels(co3)[0] AS o_label, co3.key AS o_key
  UNION
  WITH a,co2 WHERE a IS NOT NULL AND co2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCountry' AS p, labels(co2)[0] AS o_label, co2.key AS o_key
}}
RETURN DISTINCT s_label, s_key, p, o_label, o_key
'''.strip()


def compile_difference_documentcheck(qg):
    return f'''
{SCOPE_MATCH_EXCLUDE}
OPTIONAL MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)
OPTIONAL MATCH (dc)-[:requestedDocument]->(di:DocumentInstance)
WITH e,a,c,co,c2,co2,co3,dc,di,di.documentType AS di_type
WHERE (size($doc_types)=0 OR di_type IN $doc_types)
CALL (e,dc,di,di_type,a,c,co,c2,co2,co3) {{
  WITH e,dc WHERE e IS NOT NULL AND dc IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasDocumentCheck' AS p, labels(dc)[0] AS o_label, dc.key AS o_key
  UNION
  WITH dc,di WHERE dc IS NOT NULL AND di IS NOT NULL
  RETURN DISTINCT labels(dc)[0] AS s_label, dc.key AS s_key, 'requestedDocument' AS p, labels(di)[0] AS o_label, di.key AS o_key
  UNION
  WITH di,di_type WHERE di IS NOT NULL AND di_type IS NOT NULL
  RETURN DISTINCT labels(di)[0] AS s_label, di.key AS s_key, 'documentType' AS p, 'Literal' AS o_label, di_type AS o_key
  UNION
  WITH e,c WHERE e IS NOT NULL AND c IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCity' AS p, labels(c)[0] AS o_label, c.key AS o_key
  UNION
  WITH e,a WHERE e IS NOT NULL AND a IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atAirport' AS p, labels(a)[0] AS o_label, a.key AS o_key
  UNION
  WITH e,co WHERE e IS NOT NULL AND co IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCountry' AS p, labels(co)[0] AS o_label, co.key AS o_key
  UNION
  WITH a,c2 WHERE a IS NOT NULL AND c2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCity' AS p, labels(c2)[0] AS o_label, c2.key AS o_key
  UNION
  WITH c,co3 WHERE c IS NOT NULL AND co3 IS NOT NULL
  RETURN DISTINCT labels(c)[0] AS s_label, c.key AS s_key, 'locatedInCountry' AS p, labels(co3)[0] AS o_label, co3.key AS o_key
  UNION
  WITH a,co2 WHERE a IS NOT NULL AND co2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCountry' AS p, labels(co2)[0] AS o_label, co2.key AS o_key
}}
RETURN DISTINCT s_label, s_key, p, o_label, o_key
'''.strip()


def compile_questioning_cypher(qg):
    return f'''
{SCOPE_MATCH}
MATCH (e)-[:hasQuestioning]->(q:Questioning)
WHERE (size($topic_keys)=0 OR q.topicKey IN $topic_keys)
WITH e,a,c,co,c2,co2,co3,q,q.topicKey AS q_topic
CALL (e,q,q_topic,a,c,co,c2,co2,co3) {{
  WITH e,q WHERE e IS NOT NULL AND q IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasQuestioning' AS p, labels(q)[0] AS o_label, q.key AS o_key
  UNION
  WITH q,q_topic WHERE q IS NOT NULL AND q_topic IS NOT NULL
  RETURN DISTINCT labels(q)[0] AS s_label, q.key AS s_key, 'topicKey' AS p, 'Literal' AS o_label, q_topic AS o_key
  UNION
  WITH e,c WHERE e IS NOT NULL AND c IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCity' AS p, labels(c)[0] AS o_label, c.key AS o_key
  UNION
  WITH e,a WHERE e IS NOT NULL AND a IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atAirport' AS p, labels(a)[0] AS o_label, a.key AS o_key
  UNION
  WITH e,co WHERE e IS NOT NULL AND co IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCountry' AS p, labels(co)[0] AS o_label, co.key AS o_key
  UNION
  WITH a,c2 WHERE a IS NOT NULL AND c2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCity' AS p, labels(c2)[0] AS o_label, c2.key AS o_key
  UNION
  WITH c,co3 WHERE c IS NOT NULL AND co3 IS NOT NULL
  RETURN DISTINCT labels(c)[0] AS s_label, c.key AS s_key, 'locatedInCountry' AS p, labels(co3)[0] AS o_label, co3.key AS o_key
  UNION
  WITH a,co2 WHERE a IS NOT NULL AND co2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCountry' AS p, labels(co2)[0] AS o_label, co2.key AS o_key
}}
RETURN DISTINCT s_label, s_key, p, o_label, o_key
'''.strip()
def compile_biometric_cypher(qg):
    return f'''
{SCOPE_MATCH}
MATCH (e)-[:hasBiometricCheck]->(b:BiometricCheck)
WHERE (size($biometric_modalities)=0 OR b.biometricModality IN $biometric_modalities)
WITH e,a,c,co,c2,co2,co3,b,b.biometricModality AS b_mod
CALL (e,b,b_mod,a,c,co,c2,co2,co3) {{
  WITH e,b WHERE e IS NOT NULL AND b IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasBiometricCheck' AS p, labels(b)[0] AS o_label, b.key AS o_key
  UNION
  WITH b,b_mod WHERE b IS NOT NULL AND b_mod IS NOT NULL
  RETURN DISTINCT labels(b)[0] AS s_label, b.key AS s_key, 'biometricModality' AS p, 'Literal' AS o_label, b_mod AS o_key
  UNION
  WITH e,c WHERE e IS NOT NULL AND c IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCity' AS p, labels(c)[0] AS o_label, c.key AS o_key
  UNION
  WITH e,a WHERE e IS NOT NULL AND a IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atAirport' AS p, labels(a)[0] AS o_label, a.key AS o_key
  UNION
  WITH e,co WHERE e IS NOT NULL AND co IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCountry' AS p, labels(co)[0] AS o_label, co.key AS o_key
  UNION
  WITH a,c2 WHERE a IS NOT NULL AND c2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCity' AS p, labels(c2)[0] AS o_label, c2.key AS o_key
  UNION
  WITH c,co3 WHERE c IS NOT NULL AND co3 IS NOT NULL
  RETURN DISTINCT labels(c)[0] AS s_label, c.key AS s_key, 'locatedInCountry' AS p, labels(co3)[0] AS o_label, co3.key AS o_key
  UNION
  WITH a,co2 WHERE a IS NOT NULL AND co2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCountry' AS p, labels(co2)[0] AS o_label, co2.key AS o_key
}}
RETURN DISTINCT s_label, s_key, p, o_label, o_key
'''.strip()
def compile_doc_given_topic(qg):
    return f'''
{SCOPE_MATCH}
MATCH (e)-[:hasQuestioning]->(q:Questioning)
WHERE (size($topic_keys)=0 OR q.topicKey IN $topic_keys)
MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)
WHERE (size($doc_types)=0 OR di.documentType IN $doc_types)
WITH e,a,c,co,c2,co2,co3,dc,di,di.documentType AS di_type,q,q.topicKey AS q_topic
CALL (e,dc,di,di_type,q,q_topic,a,c,co,c2,co2,co3) {{
  WITH e,dc WHERE e IS NOT NULL AND dc IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasDocumentCheck' AS p, labels(dc)[0] AS o_label, dc.key AS o_key
  UNION
  WITH dc,di WHERE dc IS NOT NULL AND di IS NOT NULL
  RETURN DISTINCT labels(dc)[0] AS s_label, dc.key AS s_key, 'requestedDocument' AS p, labels(di)[0] AS o_label, di.key AS o_key
  UNION
  WITH di,di_type WHERE di IS NOT NULL AND di_type IS NOT NULL
  RETURN DISTINCT labels(di)[0] AS s_label, di.key AS s_key, 'documentType' AS p, 'Literal' AS o_label, di_type AS o_key
  UNION
  WITH e,q WHERE e IS NOT NULL AND q IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasQuestioning' AS p, labels(q)[0] AS o_label, q.key AS o_key
  UNION
  WITH q,q_topic WHERE q IS NOT NULL AND q_topic IS NOT NULL
  RETURN DISTINCT labels(q)[0] AS s_label, q.key AS s_key, 'topicKey' AS p, 'Literal' AS o_label, q_topic AS o_key
  UNION
  WITH e,c WHERE e IS NOT NULL AND c IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCity' AS p, labels(c)[0] AS o_label, c.key AS o_key
  UNION
  WITH e,a WHERE e IS NOT NULL AND a IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atAirport' AS p, labels(a)[0] AS o_label, a.key AS o_key
  UNION
  WITH e,co WHERE e IS NOT NULL AND co IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCountry' AS p, labels(co)[0] AS o_label, co.key AS o_key
  UNION
  WITH a,c2 WHERE a IS NOT NULL AND c2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCity' AS p, labels(c2)[0] AS o_label, c2.key AS o_key
  UNION
  WITH c,co3 WHERE c IS NOT NULL AND co3 IS NOT NULL
  RETURN DISTINCT labels(c)[0] AS s_label, c.key AS s_key, 'locatedInCountry' AS p, labels(co3)[0] AS o_label, co3.key AS o_key
  UNION
  WITH a,co2 WHERE a IS NOT NULL AND co2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCountry' AS p, labels(co2)[0] AS o_label, co2.key AS o_key
}}
RETURN DISTINCT s_label, s_key, p, o_label, o_key
'''.strip()
def compile_topic_given_doc(qg):
    return f'''
{SCOPE_MATCH}
MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)
WHERE (size($doc_types)=0 OR di.documentType IN $doc_types)
OPTIONAL MATCH (e)-[:hasQuestioning]->(q:Questioning)
WITH e,a,c,co,c2,co2,co3,dc,di,di.documentType AS di_type,q,q.topicKey AS q_topic
WHERE (q IS NULL OR size($topic_keys)=0 OR q_topic IN $topic_keys)
CALL (e,dc,di,di_type,q,q_topic,a,c,co,c2,co2,co3) {{
  WITH e,q WHERE e IS NOT NULL AND q IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasQuestioning' AS p, labels(q)[0] AS o_label, q.key AS o_key
  UNION
  WITH q,q_topic WHERE q IS NOT NULL AND q_topic IS NOT NULL
  RETURN DISTINCT labels(q)[0] AS s_label, q.key AS s_key, 'topicKey' AS p, 'Literal' AS o_label, q_topic AS o_key
  UNION
  WITH e,dc WHERE e IS NOT NULL AND dc IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasDocumentCheck' AS p, labels(dc)[0] AS o_label, dc.key AS o_key
  UNION
  WITH dc,di WHERE dc IS NOT NULL AND di IS NOT NULL
  RETURN DISTINCT labels(dc)[0] AS s_label, dc.key AS s_key, 'requestedDocument' AS p, labels(di)[0] AS o_label, di.key AS o_key
  UNION
  WITH di,di_type WHERE di IS NOT NULL AND di_type IS NOT NULL
  RETURN DISTINCT labels(di)[0] AS s_label, di.key AS s_key, 'documentType' AS p, 'Literal' AS o_label, di_type AS o_key
  UNION
  WITH e,c WHERE e IS NOT NULL AND c IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCity' AS p, labels(c)[0] AS o_label, c.key AS o_key
  UNION
  WITH e,a WHERE e IS NOT NULL AND a IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atAirport' AS p, labels(a)[0] AS o_label, a.key AS o_key
  UNION
  WITH e,co WHERE e IS NOT NULL AND co IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCountry' AS p, labels(co)[0] AS o_label, co.key AS o_key
  UNION
  WITH a,c2 WHERE a IS NOT NULL AND c2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCity' AS p, labels(c2)[0] AS o_label, c2.key AS o_key
  UNION
  WITH c,co3 WHERE c IS NOT NULL AND co3 IS NOT NULL
  RETURN DISTINCT labels(c)[0] AS s_label, c.key AS s_key, 'locatedInCountry' AS p, labels(co3)[0] AS o_label, co3.key AS o_key
  UNION
  WITH a,co2 WHERE a IS NOT NULL AND co2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCountry' AS p, labels(co2)[0] AS o_label, co2.key AS o_key
}}
RETURN DISTINCT s_label, s_key, p, o_label, o_key
'''.strip()
def compile_intersect_doc_questioning(qg):
    return f'''
{SCOPE_MATCH}
MATCH (e)-[:hasQuestioning]->(q:Questioning)
MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)
WITH e,a,c,co,c2,co2,co3,dc,di,di.documentType AS di_type,q,q.topicKey AS q_topic
CALL (e,dc,di,di_type,q,q_topic,a,c,co,c2,co2,co3) {{
  WITH e,dc WHERE e IS NOT NULL AND dc IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasDocumentCheck' AS p, labels(dc)[0] AS o_label, dc.key AS o_key
  UNION
  WITH dc,di WHERE dc IS NOT NULL AND di IS NOT NULL
  RETURN DISTINCT labels(dc)[0] AS s_label, dc.key AS s_key, 'requestedDocument' AS p, labels(di)[0] AS o_label, di.key AS o_key
  UNION
  WITH di,di_type WHERE di IS NOT NULL AND di_type IS NOT NULL
  RETURN DISTINCT labels(di)[0] AS s_label, di.key AS s_key, 'documentType' AS p, 'Literal' AS o_label, di_type AS o_key
  UNION
  WITH e,q WHERE e IS NOT NULL AND q IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasQuestioning' AS p, labels(q)[0] AS o_label, q.key AS o_key
  UNION
  WITH q,q_topic WHERE q IS NOT NULL AND q_topic IS NOT NULL
  RETURN DISTINCT labels(q)[0] AS s_label, q.key AS s_key, 'topicKey' AS p, 'Literal' AS o_label, q_topic AS o_key
  UNION
  WITH e,c WHERE e IS NOT NULL AND c IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCity' AS p, labels(c)[0] AS o_label, c.key AS o_key
  UNION
  WITH e,a WHERE e IS NOT NULL AND a IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atAirport' AS p, labels(a)[0] AS o_label, a.key AS o_key
  UNION
  WITH e,co WHERE e IS NOT NULL AND co IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCountry' AS p, labels(co)[0] AS o_label, co.key AS o_key
  UNION
  WITH a,c2 WHERE a IS NOT NULL AND c2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCity' AS p, labels(c2)[0] AS o_label, c2.key AS o_key
  UNION
  WITH c,co3 WHERE c IS NOT NULL AND co3 IS NOT NULL
  RETURN DISTINCT labels(c)[0] AS s_label, c.key AS s_key, 'locatedInCountry' AS p, labels(co3)[0] AS o_label, co3.key AS o_key
  UNION
  WITH a,co2 WHERE a IS NOT NULL AND co2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCountry' AS p, labels(co2)[0] AS o_label, co2.key AS o_key
}}
RETURN DISTINCT s_label, s_key, p, o_label, o_key
'''.strip()
def compile_to_cypher(qg):
    mode = qg.get('mode')
    intents = qg.get('intents') or ['documentCheck']
    if mode == 'aggregate':
        return compile_aggregate_cypher(qg)
    if mode == 'conditional':
        if qg.get('conditional') == 'topic_given_doc':
            return compile_topic_given_doc(qg)
        return compile_doc_given_topic(qg)
    if mode == 'difference' and 'documentCheck' in set(intents):
        return compile_difference_documentcheck(qg)
    if mode == 'intersect' and {'documentCheck','questioning'} <= set(intents):
        return compile_intersect_doc_questioning(qg)
    if len(intents) == 1:
        if intents[0] == 'documentCheck':
            return compile_documentcheck_cypher(qg)
        if intents[0] == 'questioning':
            return compile_questioning_cypher(qg)
        if intents[0] == 'biometric':
            return compile_biometric_cypher(qg)
    return None

cypher = compile_to_cypher(QUERY_GRAPH)
cypher





"MATCH (e:Encounter)\nOPTIONAL MATCH (e)-[:atAirport]->(a:Airport)\nOPTIONAL MATCH (e)-[:atCity]->(c:City)\nOPTIONAL MATCH (e)-[:atCountry]->(co:Country)\nOPTIONAL MATCH (a)-[:locatedInCity]->(c2:City)\nOPTIONAL MATCH (a)-[:locatedInCountry]->(co2:Country)\nOPTIONAL MATCH (c)-[:locatedInCountry]->(co3:Country)\nWITH e,a,c,co,c2,co2,co3,\n     (size($airports) > 0) AS has_airports,\n     (size($cities) > 0) AS has_cities,\n     (size($countries) > 0) AS has_countries,\n     (a IS NOT NULL AND a.key IN $airports) AS match_airport,\n     ((c IS NOT NULL AND c.key IN $cities) OR (c2 IS NOT NULL AND c2.key IN $cities)) AS match_city,\n     ((co IS NOT NULL AND co.key IN $countries) OR (co2 IS NOT NULL AND co2.key IN $countries) OR (co3 IS NOT NULL AND co3.key IN $countries)) AS match_country\nWHERE CASE\n    WHEN (has_airports OR has_cities) THEN (match_airport OR match_city)\n    WHEN has_countries THEN match_country\n    ELSE true\nEND\n\nOPTIONAL MATCH (e)-[:hasDocumentCheck]->(dc:Docume

In [463]:
# 8) Execute retrieval
params = {
    'cities': QUERY_GRAPH['scope']['cities'],
    'airports': QUERY_GRAPH['scope']['airports'],
    'countries': QUERY_GRAPH['scope']['countries'],
    'ex_cities': QUERY_GRAPH['exclude_scope']['cities'],
    'ex_airports': QUERY_GRAPH['exclude_scope']['airports'],
    'ex_countries': QUERY_GRAPH['exclude_scope']['countries'],
    'doc_types': QUERY_GRAPH['doc_types'],
    'topic_keys': QUERY_GRAPH['topic_keys'],
    'biometric_modalities': QUERY_GRAPH['biometric_modalities'],
}

rows = []
if QUERY_GRAPH['mode'] == 'union' and len(QUERY_GRAPH['intents']) > 1:
    seen = set()
    for intent in QUERY_GRAPH['intents']:
        qg_one = dict(QUERY_GRAPH)
        qg_one['intents'] = [intent]
        qg_one['mode'] = 'single'
        c = compile_to_cypher(qg_one)
        for r in run_cypher(c, params):
            t = (r.get('s_label'), r.get('s_key'), r.get('p'), r.get('o_label'), r.get('o_key'))
            if t not in seen:
                seen.add(t)
                rows.append(r)
else:
    rows = run_cypher(cypher, params) if cypher else []

rows[:70]


[{'s_label': 'Encounter',
  's_key': 'encounter_3902451_author_1',
  'p': 'hasDocumentCheck',
  'o_label': 'DocumentCheck',
  'o_key': 'doccheck_encounter_3902451_author_1_foreign_passport'},
 {'s_label': 'DocumentCheck',
  's_key': 'doccheck_encounter_3902451_author_1_foreign_passport',
  'p': 'requestedDocument',
  'o_label': 'DocumentInstance',
  'o_key': 'documentinstance_encounter_3902451_author_1_foreign_passport'},
 {'s_label': 'DocumentInstance',
  's_key': 'documentinstance_encounter_3902451_author_1_foreign_passport',
  'p': 'documentType',
  'o_label': 'Literal',
  'o_key': 'foreign_passport'},
 {'s_label': 'Encounter',
  's_key': 'encounter_3902451_author_1',
  'p': 'atCity',
  'o_label': 'City',
  'o_key': 'city_paris'},
 {'s_label': 'Encounter',
  's_key': 'encounter_3902451_author_1',
  'p': 'atAirport',
  'o_label': 'Airport',
  'o_key': 'airport_cdg'},
 {'s_label': 'Encounter',
  's_key': 'encounter_3902451_author_1',
  'p': 'atCountry',
  'o_label': 'Country',
  'o_ke

In [464]:
# 9) Postprocess / evaluate (optional)
META_KEYS = ['intents','mode','conditional','scope','exclude_scope','doc_types','topic_keys','biometric_modalities','aggregate']

def triples_to_set(rows):
    return {
        (r.get('s_label'), r.get('s_key'), r.get('p'), r.get('o_label'), r.get('o_key'))
        for r in rows
        if all(k in r for k in ['s_label','s_key','p','o_label','o_key'])
    }

def load_gold(qnum: int, gold_root: Path):
    triples = []
    metas = []
    base = gold_root / f'q_{qnum}'
    for folder in ['fr_1', 'fr_2', 'germany', 'italy', 'spain']:
        p = base / folder
        if not p.exists():
            continue
        for f in sorted(p.glob('annotation_*.json')):
            data = json.loads(f.read_text(encoding='utf-8'))
            for t in data.get('triples', []):
                s = t.get('s', {})
                o = t.get('o', {})
                triples.append((s.get('label'), s.get('key'), t.get('p'), o.get('label'), o.get('key')))
            meta = {k: data.get(k) for k in META_KEYS if k in data}
            if meta:
                metas.append(meta)
    meta_ref = metas[0] if metas else {}
    meta_consistent = all(m == meta_ref for m in metas) if metas else True

    agg_ref = None
    agg_path = base / 'aggregate_output.json'
    if agg_path.exists():
        agg_ref = json.loads(agg_path.read_text(encoding='utf-8'))

    return set(triples), meta_ref, meta_consistent, agg_ref

def norm_agg(rows):
    if rows is None:
        return rows
    out = []
    for r in rows:
        if isinstance(r, dict):
            out.append({k: r.get(k) for k in sorted(r.keys())})
        else:
            out.append(r)
    if out and isinstance(out[0], dict) and 'metric' in out[0]:
        key_fields = [k for k in out[0].keys() if k != 'metric']
        key_field = key_fields[0] if key_fields else None
        def sort_key(d):
            m = d.get('metric')
            try:
                m_val = float(m)
            except Exception:
                m_val = 0.0
            k = d.get(key_field) if key_field else ''
            return (-m_val, k)
        out = sorted(out, key=sort_key)
    return out

def norm_meta(v):
    if v == '':
        return None
    if isinstance(v, list):
        return sorted(v)
    if isinstance(v, dict):
        return {kk: sorted(vv) if isinstance(vv, list) else vv for kk, vv in v.items()}
    return v

def run_qnum(qnum: int, print_details: bool = True):
    question = get_question_text(qnum)
    route = llm_route(question, candidates)
    route = sanitize_route(route, candidates)
    route = apply_heuristics(question, route, candidates)

    query_graph = {
        'question': question,
        **route
    }

    cypher = compile_to_cypher(query_graph)
    params = {
        'cities': query_graph['scope']['cities'],
        'airports': query_graph['scope']['airports'],
        'countries': query_graph['scope']['countries'],
        'ex_cities': query_graph['exclude_scope']['cities'],
        'ex_airports': query_graph['exclude_scope']['airports'],
        'ex_countries': query_graph['exclude_scope']['countries'],
        'doc_types': query_graph['doc_types'],
        'topic_keys': query_graph['topic_keys'],
        'biometric_modalities': query_graph['biometric_modalities'],
    }

    rows = []
    if query_graph['mode'] == 'union' and len(query_graph['intents']) > 1:
        seen = set()
        for intent in query_graph['intents']:
            qg_one = dict(query_graph)
            qg_one['intents'] = [intent]
            qg_one['mode'] = 'single'
            c = compile_to_cypher(qg_one)
            for r in run_cypher(c, params):
                t = (r.get('s_label'), r.get('s_key'), r.get('p'), r.get('o_label'), r.get('o_key'))
                if t not in seen:
                    seen.add(t)
                    rows.append(r)
    else:
        rows = run_cypher(cypher, params) if cypher else []

    gold_root = ANNOT_QG_DIR if (ANNOT_QG_DIR / f'q_{qnum}').exists() else ANNOT_DIR
    gold, gold_meta, meta_consistent, gold_agg = load_gold(qnum, gold_root)

    mismatches = []
    meta_mismatches = []
    score = None

    if rows and all(k in rows[0] for k in ['s_label','s_key','p','o_label','o_key']):
        retrieved = triples_to_set(rows)
        tp = len(retrieved & gold)
        precision = tp / len(retrieved) if retrieved else 0.0
        recall = tp / len(gold) if gold else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        score = f1
        print(retrieved - gold)
        if print_details:
            print('qnum', qnum)
            print('gold_root', gold_root)
            print('retrieved', len(retrieved))
            print('gold', len(gold))
            print('precision', precision)
            print('recall', recall)
            print('f1', f1)
    else:
        if gold_agg is not None:
            ok = norm_agg(rows) == norm_agg(gold_agg)
            score = 'OK' if ok else 'НЕ ОК'
            if not ok:
                mismatches.append('aggregate_output')
        else:
            score = 'НЕ ОК'
            mismatches.append('aggregate_output(no_gold)')

        if print_details:
            print('qnum', qnum)
            print('Aggregate output:', rows[:5])
            print('gold_root', gold_root)
            print('gold', len(gold))
            if gold_agg is not None:
                ok = norm_agg(rows) == norm_agg(gold_agg)
                print('aggregate_output:', 'OK' if ok else '🟥 MISMATCH')
                if not ok:
                    print('  expected:', gold_agg)
                    print('  got     :', rows)
            else:
                print('aggregate_output: no gold')

    if print_details:
        print('meta_consistent', meta_consistent)

    if gold_meta:
        actual_meta = {k: norm_meta(query_graph.get(k)) for k in META_KEYS}
        gold_meta_norm = {k: norm_meta(gold_meta.get(k)) for k in META_KEYS}
        for k in META_KEYS:
            ok = actual_meta.get(k) == gold_meta_norm.get(k)
            if not ok:
                meta_mismatches.append(k)
            if print_details:
                print(f"{k}:", 'OK' if ok else '🟥 MISMATCH')
                if not ok:
                    print('  expected:', gold_meta_norm.get(k))
                    print('  got     :', actual_meta.get(k))
    else:
        if print_details:
            print('no gold metadata to compare')

    if print_details:
        print('')

    # overall OK flag (ignores meta mismatches)
    ok_flag = False
    if isinstance(score, float):
        ok_flag = abs(score - 1.0) < 1e-12
    elif score == 'OK':
        ok_flag = True
    if mismatches:
        ok_flag = False

    return {
        'qnum': qnum,
        'score': score,
        'mismatches': mismatches,
        'ok': ok_flag,
        'meta_mismatches': meta_mismatches,
    }

# Batch or single-run
results = []
for q in (QNUMS or []):
    results.append(run_qnum(q))

# Summary table (append/update rows in file)
if results:
    rows_out = []
    for r in results:
        status = '🟩' if r.get('ok') else '🟥'
        row = {
            'qnum': str(r['qnum']),
            'status': status,
            'score': r['score'],
            'mismatches': ', '.join(r.get('mismatches') or []),
            'meta_mismatch': ', '.join(r.get('meta_mismatches') or []),
        }
        rows_out.append(row)

    import csv
    import io
    summary_path = BASE_DIR / 'summary_qnums.csv'
    existing_rows = {}
    if summary_path.exists():
        with summary_path.open('r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                q = (row.get('qnum') or '').strip()
                if q:
                    existing_rows[q] = row

    for row in rows_out:
        existing_rows[row['qnum']] = row

    all_rows = list(existing_rows.values())
    any_mismatch = any(r.get('mismatches') for r in all_rows)
    any_meta_mismatch = any(r.get('meta_mismatch') for r in all_rows)

    cols = ['qnum', 'status', 'score']
    if any_mismatch:
        cols.append('mismatches')
    if any_meta_mismatch:
        cols.append('meta_mismatch')

    def _qnum_key(row):
        q = row.get('qnum')
        try:
            return (0, int(q))
        except Exception:
            return (1, str(q))

    all_rows = sorted(all_rows, key=_qnum_key)

    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=cols)
    writer.writeheader()
    for r in all_rows:
        writer.writerow({k: r.get(k, '') for k in cols})

    summary_path.write_text(buf.getvalue(), encoding='utf-8')

    # print csv
    print(buf.getvalue().strip())
    print(f"summary written to {summary_path}")


set()
qnum 1
gold_root /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/annotaion_q_graph
retrieved 678
gold 678
precision 1.0
recall 1.0
f1 1.0
meta_consistent True
intents: OK
mode: OK
conditional: OK
scope: OK
exclude_scope: OK
doc_types: OK
topic_keys: OK
biometric_modalities: OK
aggregate: OK

set()
qnum 2
gold_root /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/annotaion_q_graph
retrieved 1115
gold 1115
precision 1.0
recall 1.0
f1 1.0
meta_consistent True
intents: OK
mode: OK
conditional: OK
scope: OK
exclude_scope: OK
doc_types: OK
topic_keys: OK
biometric_modalities: OK
aggregate: OK

qnum 3
Aggregate output: [{'topic': 'visit_duration', 'metric': 11}, {'topic': 'visit_purpose', 'metric': 11}, {'topic': 'accommodation_details', 'metric': 8}, {'topic': 'cash_amount', 'metric': 4}, {'topic': 'itinerary_transport', 'metric': 4}]
gold_root /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эк

In [465]:
print(retrieved - gold)

NameError: name 'retrieved' is not defined

In [ ]:
20 25 27 28